## 📑 HAPS Modifications Performance

#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AYB0Hg)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

import boto3
import os

#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ"S3PREFIX" 

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

som_date = spark.sql(  
f'''
select CAST(PreviousMonthLastWorkingDate AS DATE) ReferenceDateSoM
from `{env_prefix}-cdh`.reference.ref_calendar
where CalendarDate = {run_date!r}
'''
).collect()[0]['ReferenceDateSoM']   
som_date = som_date.strftime('%Y-%m-%d')

AssignmentCutOffDate = spark.sql(
f'''
select 
CASE WHEN {run_date!r} <
(CASE WHEN DayName  = 'Saturday' 
THEN (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year({run_date!r}), '-', month({run_date!r}),'-16')))
ELSE (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year({run_date!r}), '-', month({run_date!r}),'-15')) ) END)
THEN {run_date!r}
ELSE 
(CASE WHEN DayName  = 'Saturday' 
THEN (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year({run_date!r}), '-', month({run_date!r}),'-16')))
ELSE (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year({run_date!r}), '-', month({run_date!r}),'-15')) ) END)
	END	as AssignmentCutOffDate
 FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year({run_date!r}), '-', month({run_date!r}),'-14'))
'''
).collect()[0]['AssignmentCutOffDate']   
AssignmentCutOffDate = AssignmentCutOffDate.strftime('%Y-%m-%d')

CurrentMonthLastWorkingDate = spark.sql(  
f'''
select CAST(CurrentMonthLastWorkingDate AS DATE) CurrentMonthLastWorkingDate
from `{env_prefix}-cdh`.reference.ref_calendar
where CalendarDate = {run_date!r}
'''
).collect()[0]['CurrentMonthLastWorkingDate'] 
CurrentMonthLastWorkingDate = CurrentMonthLastWorkingDate.strftime('%Y-%m-%d')

LastDateOfMonth = spark.sql(  
f'''
select CAST(LastDateOfMonth AS DATE) LastDateOfMonth
from `{env_prefix}-cdh`.reference.ref_calendar
where CalendarDate = {run_date!r}
'''
).collect()[0]['LastDateOfMonth'] 
LastDateOfMonth = LastDateOfMonth.strftime('%Y-%m-%d')

##### Schema of the table

In [0]:
table_name = 'HAPS_ModificationsPerformance'

schema = (
"BusinessDate date,"
"PortfolioName string,"
"CDI string,"
"PartyTaxRegistrationNumber string,"
"DebtorName string,"
"PostalCode string,"
"Region string,"
"AgreementKey string,"
"ExposureKey string,"
"AssetSegment string,"
"ProductKey string,"
"SoMGBVEur decimal(19,2),"
"SoMOnBalanceAmountEur decimal(19,2),"
"SoMBucket string,"
"SoMExposureStatus string,"
"IsActiveFlag string,"
"ClosingDate date,"
"MtDBucket string,"
"L3869Status string,"
"LegalProcessStatusLevel2 string,"
"OperationalLegalStatusLevel3 string,"
"LegalProcessStatusLastChangeDate date,"
"SegmentationBusinessDate date,"
"OfficialSegment string,"
"PayerFlagEntity string,"
"BorrowerTotalGBVRange string,"
"SecuredUnsecuredAccount string,"
"HighBalanceFlag smallint,"
"AssignmentSegmentDetailed string,"
"CourseOfAction string,"
"AssignmentChannel string,"
"AssignmentTeam string,"
"AssignmentAgent string,"
"SoMPipelineFlag smallint,"
"NewApplicationFlag smallint,"
"ModifiedWithinMonthFlag smallint,"
"ActivePipelineFlag smallint,"
"RejectedWithinMonthFlag smallint,"
"CancelledWithinMonthFlag smallint,"
"CompletedNotImplementedWithinMonthFlag smallint,"
"AppliedWithinQuarterFlag smallint,"
"ModifiedWithinQuarterFlag smallint,"
"AttributionChannel string,"
"AttributionTeam string,"
"AttributionAgent string,"
"FinalAssignmentChannel string,"
"FinalAssignmentTeam string,"
"FinalAssignmentAgent string,"
"StrongestActivityType string,"
"ActivityDate date,"
"AttemptFlag smallint,"
"ContactFlag smallint,"
"PositiveRPCFlag smallint,"
"NegativeRPCFlag smallint,"
"RestRPCFlag smallint,"
"ApplicationKey string,"
"ApplicationDateCreated date,"
"ApplicationActivationDate date,"
"StepCategory string,"
"StepCategoryOwner string,"
"StepSubCategory string,"
"StepSubCategoryOwner string,"
"CreditInvolvementStatus string,"
"CurrentStep string,"
"ApplicationAssigneeGroupName string,"
"CurrentStepAssignmentDate date,"
"ApplicationAssessmentDate date,"
"ApplicationImplementationStatus string,"
"AccountImplementationStatus string,"
"ApprovedFlag smallint,"
"ApplicationApprovedDate date,"
"RejectedFlag smallint,"
"RejectedDate date,"
"CancelledFlag smallint,"
"CancelledDate date,"
"ContractSigningDate date,"
"ContractSigningPlace string,"
"ContractSigningPlaceType string,"
"AppFinalStepDate date,"
"CustomerMobilePhone string,"
"ContractSigningMethod string,"
"CalculationMode string,"
"ImplementationDate date,"
"NewAccountNumber string,"
"CommunicationChannel string,"
"UserCreatedChannel string,"
"ApplicationUserCreatedFullName string,"
"ApplicationUserCreated string,"
"ClientResponse string,"
"ModificationType string,"
"SolutionType string,"
"Segment string,"
"DeclaredAffordability decimal(19,2),"
"StatedAffordability decimal(19,2),"
"IncomeUsedForCalculations decimal(19,2),"
"Source string,"
"Entity string,"
"LastActivityTypeKey string,"
"LastActivityName string,"
"LastActivityDate date,"
"TotalActivities1stWeek smallint,"
"TotalActivities2ndWeek smallint,"
"TotalActivities3rdWeek smallint,"
"TotalActivities4thWeek smallint,"
"TotalActivitiesMtD smallint,"
"LastPaymentAmountEUR decimal(19,2),"
"LastPaymentDate date,"
"CustomerTotalSoMGBVEUR decimal(19,2),"
"ContactLoanActivityGroup string,"
"ContactDebtorActivityGroup string,"
"IntramonthFtFMeeting smallint,"
"CustomerCategory string,"
"DownPaymentAmount decimal(19,2),"
"DPOCashAmount decimal(19,2),"
"AssignmentAgentUserID string,"
"AssignmentAgentUserName string,"
"AssignmentAgentAM string,"
"FinalAssignmentAgentUserID string,"
"FinalAssignmentAgentUserName string,"
"FinalAssignmentAgentAM string,"
"AttributionAgentUserID string,"
"AttributionAgentUserName string,"
"AttributionAgentAM string,"
"DialerUnsuccessfulAttemptFlag smallint,"
"DownPaymentAmountMonthFlag smallint,"
"DPOCashAmountMonthFlag smallint,"
"DownPaymentAmountQuarterFlag smallint,"
"DPOCashAmountQuarterFlag smallint,"
"InstallmentAmountCalculated decimal(19,2),"
"FirstPeriodTotalInterestRate decimal(19,2),"
"SpreadOfFirstPeriod decimal(19,2),"
"DownPaymentAmountRate decimal(19,2),"
"ModificationProduct string,"
"ImplementationStatusDescription string,"
"InsertTimeStamp timestamp"
)

In [0]:
%python
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### haps_portfolios

In [0]:
haps_portfolios = spark.sql(
f'''
SELECT DISTINCT A.PortfolioName
FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios  A
INNER JOIN (SELECT BusinessDate, MAX(IngestionTimeStamp) IngestionTimeStamp, PortfolioName 
            FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios
            GROUP BY  BusinessDate, PortfolioName) B
ON A.BusinessDate = B.BusinessDate AND A.PortfolioName = B.PortfolioName AND A.IngestionTimeStamp = B.IngestionTimeStamp
WHERE A.Report = 'HAPS_ModificationsPerformance'
AND A.IsActive = 1 
''' 
)
haps_portfolios.createOrReplaceTempView("haps_portfolios")

##### SegmentationDates 

In [0]:
SegmentationDates = spark.sql(

f'''

SELECT B.PortfolioName
      ,B.SegmentationDate
      ,MAX(A._update_timestamp) AS _update_timestamp
FROM `{env_prefix}-ptools`.assignments.offer_collection  A 
INNER JOIN (
             SELECT PortfolioName, MAX(BusinessDate) AS SegmentationDate
             FROM `{env_prefix}-ptools`.assignments.offer_collection
             WHERE BusinessDate <= CAST({AssignmentCutOffDate!r} AS DATE) 
             AND PortfolioName is NOT null 
             GROUP BY PortfolioName
           ) B
ON A.PortfolioName  = B.PortfolioName AND A.BusinessDate = B.SegmentationDate     
GROUP BY B.PortfolioName
        ,B.SegmentationDate      

'''
)
SegmentationDates.createOrReplaceTempView("SegmentationDates")

##### NonFLDTModsMtD  

In [0]:
NonFLDTModsMtD = spark.sql(
f'''
SELECT 
Portfolio,
AccountNumber, 
ATtributionChannel, 
AttributionTeam,
AttributionAgent,
ImplementationDate, 
ImplementationType,
ImplementationProduct,
cast(ImplementationDiscountAmount as decimal(19,2)) as ImplementationDiscountAmount,
cast(ImplementationAmount as decimal(19,2)) as ImplementationAmount,
cast(DownPaymentAmount as decimal(19,2)) as DownPaymentAmount
FROM  `{env_prefix}-reportinputfiles`.repoint.haps_nonfldtmodifications
WHERE last_day(ReferenceDateEOM) =  last_day({LastDateOfMonth!r})     
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) FROM `{env_prefix}-reportinputfiles`.repoint.haps_nonfldtmodifications WHERE last_day(ReferenceDateEOM) =  last_day({LastDateOfMonth!r}))                
'''
)
NonFLDTModsMtD.createOrReplaceTempView("NonFLDTModsMtD")

##### DPOMtD

In [0]:
DPOMtD = spark.sql(
f'''
SELECT 
Portfolio,
AccountNumber,
AttributionChannel,
AttributionTeam, 
AttributionAgent, 
case when lower(ImplementationDate) in('null','') then null else try_to_date(ImplementationDate, 'd/M/yyyy') end as ImplementationDate,
ImplementationType, 
ImplementationProduct,
cast(ModificationAmount as decimal(19,2)) as ModificationAmount,
cast(CashAmount as decimal(19,2)) as CashAmount
FROM  `{env_prefix}-reportinputfiles`.repoint.haps_DPO
WHERE last_day(ReferenceDateEOM) =  last_day({LastDateOfMonth!r}) 
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-reportinputfiles`.repoint.haps_DPO WHERE last_day(ReferenceDateEOM) =  last_day({LastDateOfMonth!r}))        
'''
)

DPOMtD.createOrReplaceTempView("DPOMtD")

##### Assignment

In [0]:
Assignment = spark.sql(
f'''
SELECT 
        e.BusinessDate
	, E.PortfolioName
      , E.ExposureKey                   
      , E.AgreementKey                  
      , E.CDI                           
      , E.PartyTaxRegistrationNumber    
      , E.AssetSegment
      , E.ProductKey
      , e.SoMOnBalanceAmountEUR
      , E.SoMGBVEUR
      , e.SoMBucket
      , E.SoMExposureStatus
      , e.SoMObjectiveStatusCode AS SoMObjectiveStatus
      , e.MtDIsActiveFlag
      , E.MtDCloseDate    AS MtDClosedDate
	, e.AssignmentChannel
	, e.AssignmentTeam
	, e.AssignmentAgent
      , e.QueueDescriptionFinal
	, e.CourseOfAction
      , e.AssignmentSegmentDetailed
      , e.CaseCode      
      , coalesce(concat(E.PartyTaxRegistrationNumber , '_', left(E.ExposureKey,2)), E.CDI) AS Entity
FROM `{env_prefix}-cdh-delivery`.repoint.haps_assignmentmapping    AS E 
WHERE E.BusinessDate = {run_date!r}
'''
)

Assignment.createOrReplaceTempView("Assignment")

##### AgreementLPS

In [0]:
AgreementLPS = spark.sql(
f'''
SELECT 
        e.BusinessDate
      , e.RunDate
	, E.PortfolioName
      , E.PortfolioKey   
      , e.AgreementKey
      , e.LegalProcessStatusLevel1
      , e.LegalProcessStatusLevel2
      , e.OperationalLegalStatusLevel3
      , cast( e.LegalProcessStatusLastChangeDate as date) AS LegalProcessStatusLastChangeDate
FROM `{env_prefix}-cdh`.derivative.dlegalprocessstatusagreement          AS E 
INNER JOIN haps_portfolios AS HP
ON HP.PortfolioName = E.PortfolioName                
WHERE E.BusinessDate = {run_date!r}
AND E.RunDate = (select max(RunDate) FROM `{env_prefix}-cdh`.derivative.dlegalprocessstatusagreement WHERE BusinessDate = {run_date!r} )
'''
)
AgreementLPS.createOrReplaceTempView("AgreementLPS")

##### dPartyMtd

In [0]:
dPartyMtd = spark.sql(
f'''
SELECT 
      E.BusinessDate
	, E.PortfolioName                
    , E.CDI               
    , e.PartyKey           
    , E.LegalNameGR
    , e.PartyRegion
    , e.PartyPostalCode
FROM `{env_prefix}-cdh`.derivative.dparty as E
INNER JOIN haps_portfolios AS HP
ON HP.PortfolioName = E.PortfolioName 
WHERE E.BusinessDate = {run_date!r}
AND E.SourceSystem = 'profits'	
--AND E.PartyStatus = 'Active'
and e.cdi is not null
'''
)

dPartyMtd.createOrReplaceTempView("dPartyMtd")

##### AccountsQualcoMtD

In [0]:
AccountsQualcoMtD = spark.sql(
f'''
SELECT 
      E.BusinessDate 
    , E.ExposureKey                         
    , E.Bucket
FROM `{env_prefix}-cdh`.primary.exposure as E
INNER JOIN haps_portfolios AS HP
ON HP.PortfolioName = E.PortfolioName 
where E.businessdate = {run_date!r}
AND E.SourceSystem = 'qca'
AND E.PartyRelationshipType = 'Primary Owner'
'''
)

AccountsQualcoMtD.createOrReplaceTempView("AccountsQualcoMtD")

##### Segmentation

In [0]:
Segmentation = spark.sql(
f'''
SELECT 
	  E.BusinessDate                    AS SegmentationBusinessDate
    , E.ExposureKey
	, E.segment_mapped_values        	AS OfficialSegment
	, E.danalytics__PayerFlagEntity   	AS PayerFlagEntity
	, E.Borrower_total_GBV_Range  		AS BorrowerTotalGBVRange   
    , e.Secured_flag                    AS SecuredFlag
  ,	e.HighBalanceFlag_c					AS HighBalanceFlag
FROM `{env_prefix}-ptools`.assignments.offer_collection AS E
INNER JOIN SegmentationDates as SD ON E.BusinessDate = SD.SegmentationDate AND E.PortfolioName = SD.PortfolioName and e._update_timestamp = sd._update_timestamp
WHERE E.RelationType = 'Primary Owner'

'''
)

Segmentation.createOrReplaceTempView("Segmentation")

##### FLDTApplications

In [0]:
FLDTApplications = spark.sql(
f'''
SELECT DISTINCT
		app.unifiedLoan_SpvDescription														AS PortfolioName
	,	app.customer_CustomerCode																	AS DebtorCDI
	,	app.unifiedLoan_AccountNumberWithPrefix										AS AccountNumber
	,	app.app_ApplicationKey																		AS ApplicationKey
	,	app.ApplicationImplementationStatus												AS ApplicationImplementationStatus
	-- ,	CASE
	-- 	WHEN app.profitsInfo_NewAccountNumber IS NOT NULL AND app.profitsInfo_NewAccountNumber <> ''
  	-- --WHEN app.QualcoAppDisbursedDate IS NOT NULL
	-- 	THEN 'Implemented'
	-- 	ELSE 'Not Implemented'
	-- 	END AS AccountImplementationStatus
	,	app.AccountImplementationStatus
	,	app.galaxyApplicationMetadata_ParentStepGroupingDescription							AS StepCategory
	,	app.galaxyApplicationMetadata_ParentStepGroupingOwnerName							AS StepCategoryOwner
	,	app.galaxyApplicationMetadata_StepGroupingDescription								AS StepSubCategory
	,	app.galaxyApplicationMetadata_StepGroupingOwnerName									AS StepSubCategoryOwner
	,	app.galaxyApplicationMetadata_CreditInvolvementStatusDescription					AS CreditInvolvementStatus
	,	app.app_FlowSegmentDescription														AS Segment
	,	app.customer_DeclaredAffordability													AS DeclaredAffordability
	,	app.customer_StatedAffordability													AS StatedAffordability
	,	app.customer_IncomeUsedForCalculations											AS IncomeUsedForCalculations
	,	app.galaxyApplicationMetadata_ClientResponseDescription			AS ClientResponse
	,	app.app_CommunicationChannelDescription											AS CommunicationChannel
	,	app.app_UserCreatedFullName																	AS ApplicationUserCreatedFullName
 ,	app.app_UserCreated																					AS ApplicationUserCreated
	,	app.app_DateCreated																					AS ApplicationDateCreated
	,	app.app_DateUpdated																					AS ApplicationDateUpdated
	,	app.ApplicationActivationDate																AS ApplicationActivationDate
	,	app.app_CurrentStepIntegrationCode													AS CurrentStep
	,	app.app_AssigneePortfolioGroupName													AS ApplicationAssigneeGroupName
	,	app.app_CurrentStepAssignmentDate														AS CurrentStepAssignmentDate
	,	app.product_AutoCreditStatusDescription											AS SolutionType
	,	app.product_ProductCategoryType															AS ModificationType
	,	app.QualcoAppApprovedDate																		AS AppApprovedDate
	,	app.QualcoAppPreApprovedDate																AS AppPreApprovedDate
	,	app.QualcoAppDisbursedDate																	AS AppDisbursedDate
	,	app.QualcoAppSubmittedDate																	AS AppSubmittedDate
	,	app.QualcoAppRejectedDate																	AS AppRejectedDate
	,	app.QualcoAppFinalStepDate																	AS AppFinalStepDate
	,	app.galaxyApplicationMetadata_QualcoAppCancelledDate										AS CancelledDate
	,	app.QualcoAppDisbursedDate																	AS ImplementationDate
	,	CONCAT(LEFT(app.unifiedLoan_AccountNumberWithPrefix,3),app.profitsInfo_NewAccountNumber)	AS NewAccountNumber
	,	CASE WHEN app.app_UserCreatedGroupName = 'Pilot FAs' 	THEN 'FAs'
			 WHEN app.app_UserCreatedGroupName = 'Pilot IRT' 	THEN 'IRT'
			 ELSE  app.app_UserCreatedGroupName	
    	END AS UserCreatedChannel
	,	app.app_ContactSigningPlaceName																AS ContractSigningPlace
	,	app.app_ContactSigningPlaceTypeName															AS ContractSigningPlaceType
	,	app.customer_MobilePhone																	AS CustomerMobilePhone
	,	app.app_ContractSigningMethodDescription										AS ContractSigningMethod
	,	app.modification_CalculationModeDescription									AS CalculationMode
	,	app.product_DownPaymentAmount																AS DownPaymentAmount
	, 	app.product_InstallmentAmountCalculated											as InstallmentAmountCalculated	
  	, 	app.product_FirstPeriodTotalInterestRate					as FirstPeriodTotalInterestRate
  	, 	app.product_SpreadOfFirstPeriod					as SpreadOfFirstPeriod
  	, 	app.product_DownPaymentAmountRate					as DownPaymentAmountRate
    ,	app.product_Name as ModificationProduct
    ,	APP.profitsInfo_ImplementationStatusDescription AS ImplementationStatusDescription
	, RANK() OVER (PARTITION BY app.unifiedLoan_AccountNumberWithPrefix
	 				 ORDER BY CASE WHEN app.QualcoAppDisbursedDate <= {run_date!r}  THEN 1
	 							   WHEN app.ApplicationImplementationStatus   = 'Active'	 THEN 2
	 																						 ELSE 3
	 						   END ASC,
	 						   app.ApplicationActivationDate DESC,
	 						   app.app_DateUpdated DESC	)										AS Rnk    
from `{env_prefix}-cdh-delivery`.repoint.haps_fldtallapplications AS app
WHERE APP.ActivatedApplicationFlag = 1
AND  app.BusinessDate = {run_date!r}         
'''
)

FLDTApplications.createOrReplaceTempView("FLDTApplications")

##### ApplicationsPipeline

In [0]:
ApplicationsPipeline = spark.sql(
f'''
SELECT DISTINCT 
		app.AccountNumber
	,	MAX(CASE WHEN CAST(app.ApplicationActivationDate AS DATE) <= {som_date!r}			
			 AND(   app.AccountImplementationStatus = 'Not Implemented' AND
					(app.ApplicationImplementationStatus = 'Active'	   OR
					 app.ApplicationImplementationStatus = 'Cancelled'   AND CAST(app.CancelledDate AS DATE) > {som_date!r} OR 
					 app.ApplicationImplementationStatus = 'Implemented' AND CAST(app.ApplicationDateUpdated AS DATE)> {som_date!r} )	
				  OR app.AccountImplementationStatus = 'Implemented' AND CAST(app.ImplementationDate AS DATE) > {som_date!r}
				  OR app.AccountImplementationStatus = 'Implemented' AND CAST(app.ImplementationDate AS DATE) <= {som_date!r})	THEN 1 
																																		ELSE 0 END)		AS SoMPipelineFlag
	,	MAX(CASE WHEN CAST(app.ApplicationActivationDate AS DATE) > {som_date!r}	
			 AND  CAST(app.ApplicationActivationDate AS DATE) <= {run_date!r}													THEN 1 
																																		ELSE 0 END)		AS NewApplicationFlag		
	,	MAX(CASE WHEN CAST(app.ApplicationActivationDate AS DATE) <= {run_date!r}			
			 AND( app.AccountImplementationStatus = 'Not Implemented' AND
				  (app.ApplicationImplementationStatus = 'Active'	   OR
				   app.ApplicationImplementationStatus = 'Cancelled'   AND CAST(app.CancelledDate AS DATE) > {run_date!r} OR 
				   app.ApplicationImplementationStatus = 'Implemented' AND CAST(app.ApplicationDateUpdated AS DATE)> {run_date!r})	
				  OR app.AccountImplementationStatus = 'Implemented' AND CAST(app.ImplementationDate AS DATE) > {run_date!r})	THEN 1 
																																		ELSE 0 END)		AS ActivePipelineFlag
	,	MAX(CASE WHEN app.AccountImplementationStatus = 'Implemented' AND lower(app.ModificationType) NOT like '_dpo' AND
			 CAST(app.ImplementationDate AS DATE) BETWEEN {som_date!r} AND {run_date!r}			THEN 1			
																												ELSE 0 END)		AS ModifiedWithinMonthFlag
	,	MAX(CASE WHEN app.AccountImplementationStatus = 'Not Implemented' AND
			 CAST(imp.ImplementationDate AS DATE) BETWEEN {som_date!r} AND {run_date!r}			THEN 1
																												ELSE 0 END)		AS CompletedNotImplementedWithinMonthFlag
	,	MAX(CASE WHEN app.AccountImplementationStatus = 'Implemented' AND lower(app.ModificationType) NOT like '_dpo' AND 
			 YEAR(app.ImplementationDate) = YEAR({run_date!r}) AND
	         DATE_PART('quarter', app.ImplementationDate) = DATE_PART('quarter', {run_date!r}) AND
			 CAST(app.ImplementationDate AS DATE) <= {run_date!r}										THEN 1			
																												ELSE 0 END)		AS ModifiedWithinQuarterFlag
	,	MAX(CASE WHEN YEAR(app.ApplicationActivationDate) = YEAR({run_date!r}) AND
	         DATE_PART('quarter', app.ApplicationActivationDate) = DATE_PART('quarter', {run_date!r}) AND
			 CAST(app.ApplicationActivationDate AS DATE) <= {run_date!r}								THEN 1			
																												ELSE 0 END)		AS AppliedWithinQuarterFlag
FROM FLDTApplications app
INNER JOIN Assignment SoM ON SoM.ExposureKey = app.AccountNumber
LEFT JOIN (
	SELECT DISTINCT 
				ApplicationKey
			,	MIN(ImplementationDate) ImplementationDate
	FROM FLDTApplications
	WHERE AccountImplementationStatus = 'Implemented'
	GROUP BY ApplicationKey) imp ON app.ApplicationKey = imp.ApplicationKey
GROUP BY app.AccountNumber
'''
)

ApplicationsPipeline.createOrReplaceTempView("ApplicationsPipeline")

##### Perimeter_v1

In [0]:
Perimeter_v1 = spark.sql(
f'''
	SELECT 
		ra.BusinessDate
		, ra.PortfolioName
		, ra.ExposureKey                   
		, ra.AgreementKey                  
		, ra.CDI                           
		, ra.PartyTaxRegistrationNumber    
		, ra.AssetSegment
		, ra.ProductKey
		, ra.SoMOnBalanceAmountEUR
		, ra.SoMGBVEUR
		, ra.SoMBucket
		, ra.SoMExposureStatus
		, ra.SoMObjectiveStatus
		, ra.MtDIsActiveFlag
		, ra.MtDClosedDate    
		, ra.AssignmentChannel
		, ra.AssignmentTeam
		, ra.AssignmentAgent	
		, ra.QueueDescriptionFinal						
		, ra.CourseOfAction
		, ra.AssignmentSegmentDetailed
		, ra.CaseCode    
		, seg.SegmentationBusinessDate
		, seg.OfficialSegment
		, seg.BorrowerTotalGBVRange		
		, seg.HighBalanceFlag
		, seg.PayerFlagEntity
		, seg.SecuredFlag									 
		, RA.Entity
		, IFNULL(app.SoMPipelineFlag, 0)			AS SoMPipelineFlag
		, IFNULL(app.NewApplicationFlag, 0)			AS NewApplicationFlag
		, IFNULL(app.ActivePipelineFlag, 0)			AS ActivePipelineFlag
		, IFNULL(app.ModifiedWithinMonthFlag, 0)	AS ModifiedWithinMonthFlag
		,CASE WHEN (SEG.PayerFlagEntity = 'Non_Payer' OR SEG.PayerFlagEntity is null) AND (app.ModifiedWithinMonthFlag = 1 OR app.NewApplicationFlag = 1) AND ra.AssignmentChannel = 'Legal Agencies' THEN ra.AssignmentChannel		
				WHEN (SEG.PayerFlagEntity = 'Non_Payer' OR SEG.PayerFlagEntity is null) AND (app.ModifiedWithinMonthFlag = 1 OR app.NewApplicationFlag = 1) AND FLDT.CommunicationChannel	LIKE '%Mock Credit%' THEN ra.AssignmentChannel
				WHEN (SEG.PayerFlagEntity = 'Non_Payer' OR SEG.PayerFlagEntity is null) AND (app.ModifiedWithinMonthFlag = 1 OR app.NewApplicationFlag = 1) AND FLDT.CommunicationChannel	LIKE '%Hubs%' THEN 'FAs'
				WHEN (SEG.PayerFlagEntity = 'Non_Payer' OR SEG.PayerFlagEntity is null) AND (app.ModifiedWithinMonthFlag = 1 OR app.NewApplicationFlag = 1) AND FLDT.CommunicationChannel	LIKE '%Call Center%' THEN 'IRT'
				WHEN (SEG.PayerFlagEntity = 'Non_Payer' OR SEG.PayerFlagEntity is null) AND (app.ModifiedWithinMonthFlag = 1 OR app.NewApplicationFlag = 1) THEN FLDT.CommunicationChannel			
																													ELSE NULL	END				AS AttributionChannelOriginal
		,CASE WHEN (SEG.PayerFlagEntity = 'Non_Payer' OR SEG.PayerFlagEntity is null) AND (app.ModifiedWithinMonthFlag = 1 OR app.NewApplicationFlag = 1) AND ra.AssignmentChannel = 'Legal Agencies' THEN ra.AssignmentTeam	
				WHEN (SEG.PayerFlagEntity = 'Non_Payer' OR SEG.PayerFlagEntity is null) AND (app.ModifiedWithinMonthFlag = 1 OR app.NewApplicationFlag = 1) AND FLDT.CommunicationChannel	LIKE '%Mock Credit%' THEN ra.AssignmentTeam
				WHEN (SEG.PayerFlagEntity = 'Non_Payer' OR SEG.PayerFlagEntity is null) AND (app.ModifiedWithinMonthFlag = 1 OR app.NewApplicationFlag = 1) THEN FLDT.UserCreatedChannel			
																													ELSE NULL	END				AS AttributionTeamOriginal
		,CASE WHEN (SEG.PayerFlagEntity = 'Non_Payer' OR SEG.PayerFlagEntity is null) AND (app.ModifiedWithinMonthFlag = 1 OR app.NewApplicationFlag = 1) AND ra.AssignmentChannel = 'Legal Agencies' THEN ra.AssignmentAgent	
				WHEN (SEG.PayerFlagEntity = 'Non_Payer' OR SEG.PayerFlagEntity is null) AND (app.ModifiedWithinMonthFlag = 1 OR app.NewApplicationFlag = 1) AND FLDT.CommunicationChannel	LIKE '%Mock Credit%' THEN ra.AssignmentAgent
				WHEN (SEG.PayerFlagEntity = 'Non_Payer' OR SEG.PayerFlagEntity is null) AND (app.ModifiedWithinMonthFlag = 1 OR app.NewApplicationFlag = 1) THEN FLDT.ApplicationUserCreatedFullName		
																												ELSE NULL	END				AS AttributionAgentOriginal
	FROM Assignment		          AS RA 
	LEFT JOIN Segmentation				AS SEG ON SEG.ExposureKey=  RIGHT(rA.ExposureKey, LEN(rA.ExposureKey) - 3)
	LEFT JOIN ApplicationsPipeline		    AS app	ON ra.ExposureKey = APP.AccountNumber
	LEFT JOIN FLDTApplications			AS FLDT	ON ra.ExposureKey = FLDT.AccountNumber	and FLDT.rnk = 1
'''
)

Perimeter_v1.createOrReplaceTempView("Perimeter_v1")

##### Perimeter_v4

In [0]:
Perimeter_v4 = spark.sql(
f'''
SELECT  
		P.*
	,	CASE WHEN D.AccountNumber IS NOT NULL                                               THEN D.AttributionTeam --CAST( D.AttributionTeam AS STRING)
          WHEN M.AccountNumber IS NOT NULL AND p.MtDIsActiveFlag  = false           THEN m.AttributionTeam-- CAST( m.AttributionTeam AS STRING)
          WHEN (p.PayerFlagEntity = 'Non_Payer' OR p.PayerFlagEntity is null) AND (p.ModifiedWithinMonthFlag = 1 OR p.NewApplicationFlag = 1) AND P.QueueDescriptionFinal  LIKE 'ΔΤ3869[_]%' 
 					and p.AttributionChannelOriginal LIKE '%FAs%'						 THEN p.AttributionTeamOriginal 
 			WHEN (p.PayerFlagEntity = 'Non_Payer' OR p.PayerFlagEntity is null) AND (p.ModifiedWithinMonthFlag = 1 OR p.NewApplicationFlag = 1) AND P.QueueDescriptionFinal  LIKE 'ΔΤ3869[_]%' 
 					and p.AttributionChannelOriginal NOT LIKE '%FAs%'					 THEN NULL
																						 ELSE p.AttributionTeamOriginal 
		END AS AttributionTeamFinal
	,	CASE WHEN D.AccountNumber IS NOT NULL                                               THEN D.AttributionAgent
         WHEN M.AccountNumber IS NOT NULL AND p.MtDIsActiveFlag  = false          THEN m.AttributionAgent
         WHEN (p.PayerFlagEntity = 'Non_Payer' OR p.PayerFlagEntity is null) AND (p.ModifiedWithinMonthFlag = 1 OR p.NewApplicationFlag = 1) AND P.QueueDescriptionFinal  LIKE 'ΔΤ3869[_]%' 
 					and p.AttributionChannelOriginal LIKE '%FAs%'						 THEN p.AttributionAgentOriginal 
 			WHEN (p.PayerFlagEntity = 'Non_Payer' OR p.PayerFlagEntity is null) AND (p.ModifiedWithinMonthFlag = 1 OR p.NewApplicationFlag = 1) AND P.QueueDescriptionFinal  LIKE 'ΔΤ3869[_]%' 
 					and p.AttributionChannelOriginal NOT LIKE '%FAs%'					 THEN NULL
																						 ELSE p.AttributionAgentOriginal
		END AS AttributionAgentFinal
	,	CASE WHEN D.AccountNumber IS NOT NULL                                               THEN D.AttributionChannel
         WHEN M.AccountNumber IS NOT NULL AND p.MtDIsActiveFlag  = false       THEN m.AttributionChannel
         WHEN (p.PayerFlagEntity = 'Non_Payer' OR p.PayerFlagEntity is null) AND (p.ModifiedWithinMonthFlag = 1 OR p.NewApplicationFlag = 1) AND P.QueueDescriptionFinal  LIKE 'ΔΤ3869[_]%' 
 																						 THEN 'L3869 ΔΤ - FAs'
 																						 ELSE p.AttributionChannelOriginal 
		END AS AttributionChannelFinal
  ,	CASE WHEN D.AccountNumber IS NOT NULL       THEN 1
         WHEN M.AccountNumber IS NOT NULL AND p.MtDIsActiveFlag  = false       THEN 1
    else 0 END AS IsExternalMod
FROM Perimeter_v1	AS P
LEFT JOIN NonFLDTModsMtD as M ON RIGHT(p.ExposureKey, LEN(p.ExposureKey)-3) = CASE WHEN LEFT(CAST(M.AccountNumber AS STRING),1) RLIKE '[A-Z]' THEN RIGHT(CAST(M.AccountNumber AS STRING), LEN(CAST(M.AccountNumber AS STRING))-3) ELSE CAST(M.AccountNumber AS STRING) END
LEFT JOIN DPOMtD as D ON RIGHT(p.ExposureKey, LEN(p.ExposureKey)-3) = CASE WHEN LEFT(CAST(D.AccountNumber AS STRING),1) RLIKE '[A-Z]' THEN RIGHT(CAST(D.AccountNumber AS STRING), LEN(CAST(D.AccountNumber AS STRING))-3) ELSE CAST(D.AccountNumber AS STRING) END
UNION ALL 
SELECT 
    {run_date!r} as BusinessDate
  , D.Portfolio
  , D.AccountNumber                   
  , NULL AS AgreementKey                  
  , NULL AS CDI                           
  , NULL AS PartyTaxRegistrationNumber    
  , NULL AS AssetSegment
  , NULL AS ProductKey
  , NULL AS SoMOnBalanceAmountEUR
  , NULL AS SoMGBVEUR
  , NULL AS SoMBucket
  , NULL AS SoMExposureStatus
  , NULL AS SoMObjectiveStatus
  , NULL AS MtDIsActiveFlag
  , NULL AS MtDClosedDate    
  , NULL AS AssignmentChannel
  , NULL AS AssignmentTeam
  , NULL AS AssignmentAgent	
  , NULL AS QueueDescriptionFinal						
  , NULL AS CourseOfAction
  , NULL AS AssignmentSegmentDetailed
  , NULL AS CaseCode    
  , NULL AS SegmentationBusinessDate
  , NULL AS OfficialSegment
  , NULL AS BorrowerTotalGBVRange		
  , NULL AS HighBalanceFlag
  , NULL AS PayerFlagEntity
  , NULL AS SecuredFlag									 
  , NULL AS Entity
  , NULL AS SoMPipelineFlag
  , NULL AS NewApplicationFlag
  , NULL AS ActivePipelineFlag
  , 1 AS ModifiedWithinMonthFlag
  , D.AttributionChannel AS AttributionChannelOriginal
	, D.AttributionTeam AS AttributionTeamOriginal
	, D.AttributionAgent AS AttributionAgentOriginal
 	,	D.AttributionTeam as AttributionTeamFinal
	,	D.AttributionAgent as AttributionAgentFinal
	,	D.AttributionChannel as AttributionChannelFinal
  , 1 as IsExternalMod
  From DPOMtD AS D
  LEFT JOIN Assignment as SOM ON RIGHT(SoM.ExposureKey, LEN(SoM.ExposureKey)-3) = CASE WHEN LEFT(CAST(D.AccountNumber AS STRING),1) RLIKE '[A-Z]' THEN RIGHT(CAST(D.AccountNumber AS STRING), LEN(CAST(D.AccountNumber AS STRING))-3) ELSE CAST(D.AccountNumber AS STRING) END
  WHERE SoM.ExposureKey IS NULL 
UNION ALL 
SELECT 
    {run_date!r} as BusinessDate
  , D.Portfolio
  , D.AccountNumber                   
  , NULL AS AgreementKey                  
  , NULL AS CDI                           
  , NULL AS PartyTaxRegistrationNumber    
  , NULL AS AssetSegment
  , NULL AS ProductKey
  , NULL AS SoMOnBalanceAmountEUR
  , NULL AS SoMGBVEUR
  , NULL AS SoMBucket
  , NULL AS SoMExposureStatus
  , NULL AS SoMObjectiveStatus
  , NULL AS MtDIsActiveFlag
  , NULL AS MtDClosedDate    
  , NULL AS AssignmentChannel
  , NULL AS AssignmentTeam
  , NULL AS AssignmentAgent	
  , NULL AS QueueDescriptionFinal						
  , NULL AS CourseOfAction
  , NULL AS AssignmentSegmentDetailed
  , NULL AS CaseCode    
  , NULL AS SegmentationBusinessDate
  , NULL AS OfficialSegment
  , NULL AS BorrowerTotalGBVRange		
  , NULL AS HighBalanceFlag
  , NULL AS PayerFlagEntity
  , NULL AS SecuredFlag									 
  , NULL AS Entity
  , NULL AS SoMPipelineFlag
  , NULL AS NewApplicationFlag
  , NULL AS ActivePipelineFlag
  , 1 AS ModifiedWithinMonthFlag
  , D.AttributionChannel AS AttributionChannelOriginal
	, D.AttributionTeam AS AttributionTeamOriginal
	, D.AttributionAgent AS AttributionAgentOriginal
 	,	D.AttributionTeam as AttributionTeamFinal
	,	D.AttributionAgent as AttributionAgentFinal
	,	D.AttributionChannel as AttributionChannelFinal
  , 1 as IsExternalMod
  From NonFLDTModsMtD AS D
  LEFT JOIN Assignment as SOM ON RIGHT(SoM.ExposureKey, LEN(SoM.ExposureKey)-3) = CASE WHEN LEFT(CAST(D.AccountNumber AS STRING),1) RLIKE '[A-Z]' THEN RIGHT(CAST(D.AccountNumber AS STRING), LEN(CAST(D.AccountNumber AS STRING))-3) ELSE CAST(D.AccountNumber AS STRING) END
  WHERE SoM.ExposureKey IS NULL 
'''
)

Perimeter_v4.createOrReplaceTempView("Perimeter_v4")

##### Perimeter_v5

In [0]:
Perimeter_v5 = spark.sql(
f'''
SELECT  
		P.*
  ,	CASE WHEN {CurrentMonthLastWorkingDate!r} = {run_date!r} AND p.IsExternalMod = 1 THEN P.AttributionChannelFinal
		WHEN {CurrentMonthLastWorkingDate!r} = {run_date!r} AND (p.PayerFlagEntity = 'Non_Payer' OR p.PayerFlagEntity is null) AND (p.ModifiedWithinMonthFlag = 1 OR p.NewApplicationFlag = 1) THEN P.AttributionChannelFinal
																																																																																 ELSE P.ASsignmentChannel
		END AS ASsignmentChannelFinal
   ,	CASE WHEN {CurrentMonthLastWorkingDate!r} = {run_date!r} AND p.IsExternalMod = 1 THEN P.AttributionTeamFinal 
   			 WHEN {CurrentMonthLastWorkingDate!r} = {run_date!r} AND (p.PayerFlagEntity = 'Non_Payer' OR p.PayerFlagEntity is null) AND (p.ModifiedWithinMonthFlag = 1 OR p.NewApplicationFlag = 1) THEN P.AttributionTeamFinal
																																																																																 ELSE P.AssignmentTeam
		END AS AssignmentTeamFinal 
  ,	CASE WHEN {CurrentMonthLastWorkingDate!r} = {run_date!r} AND p.IsExternalMod = 1 THEN P.AttributionAgentFinal 
  		 WHEN {CurrentMonthLastWorkingDate!r} = {run_date!r} AND (p.PayerFlagEntity = 'Non_Payer' OR p.PayerFlagEntity is null) AND (p.ModifiedWithinMonthFlag = 1 OR p.NewApplicationFlag = 1) THEN P.AttributionAgentFinal
																																																																																 ELSE P.AssignmentAgent
		END AS AssignmentAgentFinal	
FROM Perimeter_v4	AS P
'''
)

Perimeter_v5.createOrReplaceTempView("Perimeter_v5")

In [0]:
# %sql
# DROP TABLE IF EXISTS `{env_prefix}-cdh-delivery`.repoint.int_haps_ModificationsPerformance_Perimeter;
# CREATE TABLE `{env_prefix}-cdh-delivery`.repoint.int_haps_ModificationsPerformance_Perimeter AS
# SELECT * 
# FROM Perimeter_v5

##### Interactions_v0

In [0]:
Interactions_v0 = spark.sql(
f'''

select DISTINCT
    ia.ExposureKey
  , ia.InteractionTypeKey
  , ia.InteractionDate
  , ia.InteractionType 
  , ia.cdi
  , ia.InteractionKey
  , ia.CreatedDate  
  , rank () over (partition by ia.ExposureKey, ia.InteractionKey order by businessdate asc) as rnk
from 
`{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) ia
INNER JOIN haps_portfolios AS HP
ON HP.PortfolioName = ia.PortfolioName 
where 1=1
and ia.InteractionDate > {som_date!r}
and ia.InteractionDate <= {run_date!r}
and ia.sourcesystem = 'qca'
'''
)
Interactions_v0.createOrReplaceTempView("Interactions_v0")


##### InteractionsAll

In [0]:
InteractionsAll = spark.sql(
f'''

select distinct  
   ia.ExposureKey
  , ia.InteractionTypeKey
  , ia.InteractionDate
  , ia.InteractionType
  , ia.CreatedDate 
  , ia.cdi
  , m.Activity_Code
  , CASE WHEN left(m.`1st_level_category`,1) = '5'             THEN 4    
         WHEN left(m.`1st_level_category`,1) = '4'             THEN 4
         WHEN left(m.`1st_level_category`,1) = '3'             THEN 3
         WHEN left(m.`1st_level_category`,1) = '2'             THEN 2                      
         WHEN left(m.`2nd_level_category`,3) = '1.2'           THEN 1.2
         WHEN left(m.`2nd_level_category`,3) = '1.1'           THEN 1.1
    END AS Priority
  , CASE WHEN left(m.`1st_level_category`,1) = '5'             THEN 5    
         WHEN left(m.`1st_level_category`,1) = '4'             THEN 4
         WHEN left(m.`1st_level_category`,1) = '3'             THEN 3
         WHEN left(m.`1st_level_category`,1) = '2'             THEN 2                      
         WHEN left(m.`2nd_level_category`,3) = '1.2'           THEN 1.2
         WHEN left(m.`2nd_level_category`,3) = '1.1'           THEN 1.1
    END AS Priority_Aux    
  , m.`1st_level_category`
  , m.`2nd_level_category`
  , m.hierarchy
  , m.`3rd_level_category`
  , M.base_category
  , rank() over (partition by ia.ExposureKey, m.Activity_Group, ia.InteractionDate order by
              CASE  WHEN left(m.`1st_level_category`,1) = '5'             THEN 4    
                    WHEN left(m.`1st_level_category`,1) = '4'             THEN 4
                    WHEN left(m.`1st_level_category`,1) = '3'             THEN 3
                    WHEN left(m.`1st_level_category`,1) = '2'             THEN 2                      
                    WHEN left(m.`2nd_level_category`,3) = '1.2'           THEN 1.2
                    WHEN left(m.`2nd_level_category`,3) = '1.1'           THEN 1.1
                      END desc
                      , ia.CreatedDate desc
                      , ia.InteractionKey desc) as rank
  , m.Activity_Group
  , CASE WHEN m.`1st_level_category` IN ('1. no contact', '2. contact - Not RPC', '5.Positive Communication') THEN m.`3rd_level_category`
         WHEN m.`1st_level_category` = '3. Rest RPC Communications' THEN m.`1st_level_category`
         WHEN m.`1st_level_category` = '4. Negative Communication' THEN m.`2nd_level_category`      
    END AS ActivityTypeGroup
 -- , p.Entity
from Interactions_v0 ia
join `{env_prefix}-reportinputfiles`.repoint.core_qcabactivitiesmapping m on ia.InteractionTypeKey = m.Activity_Code 
and m.BusinessDate = (select max(BusinessDate) from `{env_prefix}-reportinputfiles`.repoint.core_qcabactivitiesmapping)   
where  
(m.hierarchy  not in (0,1) or (m.hierarchy = 0 and base_category = 'attempt'))
and ia.rnk = 1

'''
)
InteractionsAll.createOrReplaceTempView("InteractionsAll")


##### Dialer_v0

In [0]:
Dialer_v0 = spark.sql(
f'''

select DISTINCT  
      de.ExposureKey
    , cast('' as string) as InteractionTypeKey 
    , cast(DateHour as date ) as InteractionDate
    , DateHour as CreatedDate    
    , 1.15 Priority
    , 1.15 Priority_Aux 
from `{env_prefix}-cdh`.primary.dialer d
join `{env_prefix}-cdh`.primary.exposure de on cast(d.casecode as string) = cast(de.casecode as string) and d.BusinessDate = de.BusinessDate and de.sourcesystem = 'qca'
join haps_portfolios HP on HP.PortfolioName = de.PortfolioName 
and cast(DateHour as date ) > last_day({som_date!r})
and cast(DateHour as date ) <= {run_date!r}
'''
)
Dialer_v0.createOrReplaceTempView("Dialer_v0")


##### GroupedActivities

In [0]:
GroupedActivities = spark.sql(
f'''

select distinct  
   ia.ExposureKey
   , ia.Activity_Group
   , ia.InteractionDate
  , MAX(CASE WHEN IA.Priority_Aux >= 1.15 THEN 1 ELSE 0 END) AS AttemptFlag
  , MAX(CASE WHEN IA.Priority_Aux >= 2 THEN 1 ELSE 0 END) AS ContactFlag
  , MAX(CASE WHEN IA.Priority_Aux in (3,4,5) THEN 1 ELSE 0 END) AS RPCFlag
  , MAX(CASE WHEN IA.Priority_Aux = 5 THEN 1 ELSE 0 END) AS PositiveRPCFlag
  , MAX(CASE WHEN IA.Priority_Aux = 4 THEN 1 ELSE 0 END) AS NegativeRPCFlag
  , MAX(CASE WHEN IA.Priority_Aux = 3 THEN 1 ELSE 0 END) AS RestRPCFlag
from InteractionsAll AS iA 
WHERE lower(iA.Activity_Group) like '%modifications%'
group by ia.ExposureKey, ia.InteractionDate , IA.Activity_Group

'''
)
GroupedActivities.createOrReplaceTempView("GroupedActivities")


##### GroupedDialerActivities

In [0]:
GroupedDialerActivities = spark.sql(
f'''

select distinct  
   ia.ExposureKey
   , ia.InteractionDate
  , MAX(CASE WHEN IA.Priority_Aux = 1.15 THEN 1 ELSE 0 END)  AS DialerUnsuccessfulAttemptFlag
from  Dialer_v0 AS IA
group by ia.ExposureKey, ia.InteractionDate

'''
)
GroupedDialerActivities.createOrReplaceTempView("GroupedDialerActivities")


##### LoanActivitiesMtD

In [0]:
LoanActivitiesMtD = spark.sql(
f'''

SELECT 
   SoM.CDI
  , SOM.ExposureKey AS ExposureKeyPrefix
  , SoM.AssignmentChannelFinal
  , SoM.Entity
  , Act.*
  , CASE WHEN gd.DialerUnsuccessfulAttemptFlag = 1 THEN 1 ELSE g.AttemptFlag END AttemptFlag
  , gd.DialerUnsuccessfulAttemptFlag
  , g.ContactFlag
  , g.RPCFlag
  , CASE WHEN act.Priority_Aux = 5 THEN   g.PositiveRPCFlag ELSE 0 END AS PositiveRPCFlag
  , CASE WHEN act.Priority_Aux = 4 THEN   g.NegativeRPCFlag ELSE 0 END AS NegativeRPCFlag  
  , CASE WHEN act.Priority_Aux = 3 THEN   g.RestRPCFlag ELSE 0 END AS RestRPCFlag    
  --, c.ActivityTypeGroup
,	RANK() OVER(PARTITION BY SOM.ExposureKey ORDER BY act.ActivityPriority DESC, act.InteractionDate DESC, act.CreatedDate desc, act.InteractionTypeKey DESC) Rank  
,	RANK() OVER(PARTITION BY SOM.ExposureKey ORDER BY (CASE WHEN act.ActivityPriority = 1.15 THEN -1 ELSE act.ActivityPriority END)  DESC, act.InteractionDate DESC, act.CreatedDate desc, act.InteractionTypeKey DESC) RankExcludingDialer  
FROM (
SELECT 
    i.ExposureKey
  , i.InteractionTypeKey
  , i.InteractionDate
  , i.Activity_Code
  , I.Priority    AS ActivityPriority
  , i.Priority_Aux
  , i.InteractionType
  , I.Activity_Group
  , i.CreatedDate
  , CAST( i.ActivityTypeGroup AS STRING) AS ActivityTypeGroup
FROM InteractionsAll AS I
WHERE lower(I.Activity_Group) like '%modification%'
AND I.rank = 1

UNION 

SELECT 
    D.ExposureKey  
  , D.InteractionTypeKey
  , D.InteractionDate
  , NULL AS ActivityCode
  , D.Priority
  , d.Priority_Aux
  , 'Dialer' AS InteractionType
  , NULL AS Activity_Group
  , d.CreatedDate
  , CAST (NULL AS STRING) AS ActivityTypeGroup
FROM Dialer_v0 AS D ) AS Act        
INNER JOIN Perimeter_v5 AS SoM ON  Act.ExposureKey = RIGHT(SoM.ExposureKey, LEN(SoM.ExposureKey) - 3) 
-- INNER JOIN `{env_prefix}-cdh-delivery`.repoint.int_haps_ModificationsPerformance_Perimeter AS SoM ON  Act.ExposureKey = RIGHT(SoM.ExposureKey, LEN(SoM.ExposureKey) - 3) 
LEFT JOIN GroupedActivities AS G 
ON Act.ExposureKey = G.ExposureKey AND act.InteractionDate = g.InteractionDate AND ACT.Activity_Group = G.Activity_Group
LEFT JOIN GroupedDialerActivities AS GD 
ON GD.ExposureKey = Act.ExposureKey  AND act.InteractionDate = gD.InteractionDate
WHERE SoM.AssignmentChannelFinal IN ( 'FAs', 'IRT')
'''
)
LoanActivitiesMtD.createOrReplaceTempView("LoanActivitiesMtD")

In [0]:
# %sql
# DROP TABLE IF EXISTS `{env_prefix}-cdh-delivery`.repoint.int_haps_modificationsperformance_InteractionsMtD;
# CREATE TABLE `{env_prefix}-cdh-delivery`.repoint.int_haps_modificationsperformance_InteractionsMtD AS
# SELECT * 
# FROM LoanActivitiesMtD

##### StrongestDebtorActivityMtD

In [0]:
StrongestDebtorActivityMtD = spark.sql(
f'''
SELECT * 
FROM (SELECT DISTINCT 
			Entity
		,	ActivityPriority
		,	InteractionDate
		,	InteractionTypeKey
		,	ActivityTypeGroup
    	, 	AssignmentChannelFinal
     	,	RANK() OVER(PARTITION BY Entity ORDER BY ActivityPriority DESC, InteractionDate DESC, CreatedDate desc, ActivityTypeGroup DESC, InteractionTypeKey DESC)  Rank
    from LoanActivitiesMtD a --`{env_prefix}-cdh-delivery`.repoint.int_haps_modificationsperformance_InteractionsMtD a
	  WHERE a.AssignmentChannelFinal in ('FAs', 'IRT')
    and a.RankExcludingDialer = 1 and a.ActivityPriority <> 1.15
	  ) a
WHERE Rank = 1

'''
)

StrongestDebtorActivityMtD.createOrReplaceTempView("StrongestDebtorActivityMtD")

##### DebtorFtFActivityMtD

In [0]:
DebtorFtFActivityMtD = spark.sql(
f'''
SELECT DISTINCT  1 FtFActivity, SoM.Entity --, SoM.ParentPartyId
from Interactions_v0 ia
join perimeter_v5 as SoM -- `{env_prefix}-cdh-delivery`.repoint.int_haps_ModificationsPerformance_Perimeter as SoM 
ON RIGHT(SoM.ExposureKey, LEN(SoM.ExposureKey) - 3) = ia.ExposureKey
where 1=1
and ia.InteractionTypeKey in ('7024','6071')
and SoM.AssignmentChannelFinal = 'FAs'


'''
)

DebtorFtFActivityMtD.createOrReplaceTempView("DebtorFtFActivityMtD")

##### TotalActivitiesMtD

In [0]:
TotalActivitiesMtD = spark.sql(
f'''
SELECT 
        ExposureKeyPrefix,
        COUNT(DISTINCT CASE WHEN CAST(InteractionDate AS DATE) BETWEEN DATE_ADD({som_date!r}, 1) AND DATE_ADD({som_date!r}, 7) THEN CAST(InteractionDate AS DATE) ELSE NULL END) AS TotalActivities1stWeek,
        COUNT(DISTINCT CASE WHEN CAST(InteractionDate AS DATE) BETWEEN DATE_ADD({som_date!r}, 8) AND DATE_ADD({som_date!r}, 14) THEN CAST(InteractionDate AS DATE) ELSE NULL END) AS TotalActivities2ndWeek,
        COUNT(DISTINCT CASE WHEN CAST(InteractionDate AS DATE) BETWEEN DATE_ADD({som_date!r}, 15) AND DATE_ADD({som_date!r}, 21) THEN CAST(InteractionDate AS DATE) ELSE NULL END) AS TotalActivities3rdWeek,
        COUNT(DISTINCT CASE WHEN CAST(InteractionDate AS DATE) BETWEEN DATE_ADD({som_date!r}, 22) AND {CurrentMonthLastWorkingDate!r} THEN CAST(InteractionDate AS DATE) ELSE NULL END) AS TotalActivities4thWeek,
        COUNT(DISTINCT CASE WHEN CAST(InteractionDate AS DATE) BETWEEN DATE_ADD({som_date!r}, 1) AND {CurrentMonthLastWorkingDate!r} THEN CAST(InteractionDate AS DATE) ELSE NULL END) AS TotalActivitiesMtD
FROM loanactivitiesMtD a --`{env_prefix}-cdh-delivery`.repoint.int_haps_modificationsperformance_InteractionsMtD a
where a.ActivityPriority <> 1.15
GROUP BY ExposureKeyPrefix

'''
)

TotalActivitiesMtD.createOrReplaceTempView("TotalActivitiesMtD")

##### L3869Status

In [0]:
L3869Status = spark.sql(
f'''
SELECT *
FROM(
SELECT  CDI
      , InLawFlag
	  , ApplicationStatus
	  , LawStatus
	  , CalcLawStatus
	  ,	InterbankStatus3869
   	  , rank() over (partition by CDI order by BusinessDate desc) as rank
FROM `{env_prefix}-cdh`.primary.UnderLaw ul
INNER JOIN haps_portfolios AS HP
ON HP.PortfolioName = ul.PortfolioName 
WHERE BusinessDate <= {run_date!r}
AND UnderLawType = 'N3869'
AND sourcesystem = 'qca'
)
WHERE rank = 1
'''
)

L3869Status.createOrReplaceTempView("L3869Status")

##### LastPayment 

In [0]:
LastPayment = spark.sql(
f'''
SELECT      ExposureKey	 
        ,   PaymentActualDate
        ,   SUM(PaymentAmountEUR) PaymentAmountEUR
FROM (
        SELECT DISTINCT
                    P.ExposureKey
                ,   P.PaymentActualDate
                ,   P.PaymentAmount AS PaymentAmountEUR
                ,   p.paymentkey
                --,   cast(P.PaymentAmount / coalesce( E.ExchangeRate, 1)  as decimal(19,2)) AS PaymentAmountEUR  -- stin evropoiisi to join einai me to payment date
                ,   RANK() OVER(PARTITION BY P.ExposureKey ORDER by P.PaymentActualDate DESC) AS Rnk
        FROM `{env_prefix}-cdh`.primary.Payment AS P
        --LEFT JOIN `{env_prefix}-cdh`.reference.exchangerates as E ON E.OriginalCurrency = P.Currency and e.BusinessDate = p.PaymentActualDate
        WHERE P.BusinessDate = {run_date!r}
		AND P.PortfolioName IN ('Galaxy II Funding DAC', 'Orion X Securitisation DAC', 'Cosmos Retail') 
 		AND P.sourcesystem = 'profits'
                --AND P.paymenttypegroup IN (' Cash','Cash')
                AND P.PaymentType IN ('Είσπραξη από πελάτη' ,'Είσπραξη από πελάτη, κατόπιν ρύθμισης που έγινε στη Cepal.')
        AND  P.PaymentActualDate <= {run_date!r}) p
WHERE p.Rnk = 1
GROUP BY    ExposureKey
        ,   PaymentActualDate

'''
)

LastPayment.createOrReplaceTempView("LastPayment")

##### Perimeter_v6

In [0]:
Perimeter_v6 = spark.sql(
f'''
SELECT  
			
		SoM.PortfolioName
	,	SoM.CDI
	,	SoM.PartyTaxRegistrationNumber
	,	SoM.AgreementKey
	,	SoM.ExposureKey
	,	SoM.AssetSegment
	,	SoM.ProductKey
	,	SoM.SoMGBVEUR
	,	SoM.SoMOnBalanceAmountEUR
	,	SoM.SoMBucket
	,	SoM.SoMExposureStatus
	,	SoM.MtDIsActiveFlag AS IsActiveFlag
	,	SoM.MtDClosedDate AS ClosingDate
	, 	SoM.SegmentationBusinessDate				
	,	SoM.OfficialSegment
	,	SoM.PayerFlagEntity
	,	SoM.BorrowerTotalGBVRange
	,	SoM.SecuredFlag AS SecuredUnsecuredAccount
	,	SOM.HighBalanceFlag
	,	SoM.AssignmentSegmentDetailed				
	,	SoM.CourseOfAction											
	,	SoM.AssignmentChannel
	,	SoM.AssignmentTeam
	,	SoM.AssignmentAgent
	,	pip.SoMPipelineFlag
	,	pip.NewApplicationFlag
	,	pip.ModifiedWithinMonthFlag
	,	pip.ActivePipelineFlag
	,	CASE WHEN CAST(app.AppRejectedDate	AS DATE) <= {run_date!r} AND CAST(app.AppRejectedDate	AS DATE) > {som_date!r}  THEN 1 ELSE 0 END RejectedWithinMonthFlag
 	,	CASE WHEN CAST(app.CancelledDate	AS DATE) <= {run_date!r} AND CAST(app.CancelledDate	AS DATE) > {som_date!r}  THEN 1 ELSE 0 END CancelledWithinMonthFlag
	,	pip.CompletedNotImplementedWithinMonthFlag
	,	pip.AppliedWithinQuarterFlag
	,	pip.ModifiedWithinQuarterFlag
	,	SoM.AttributionChannelFinal	AS AttributionChannel				
	,	SoM.AttributionTeamFinal	AS AttributionTeam					
	,	SoM.AttributionAgentFinal	AS AttributionAgent
	,	SoM.AssignmentChannelFinal								
	,	SoM.AssignmentTeamFinal								
	,	SoM.AssignmentAgentFinal							
	,	CASE WHEN lact.Priority_aux = 1.1 	then 'No Attempt'
     		 WHEN lact.Priority_aux = 1.15 	then 'Attempt'
			 	 WHEN lact.Priority_aux = 1.2 	then 'Attempt'
       	 WHEN lact.Priority_aux = 2 	then 'Contact'
         WHEN lact.Priority_aux = 3 	then 'Rest RPC Communication'
         WHEN lact.Priority_aux = 4 	then 'Negative Communication'
         WHEN lact.Priority_aux = 5 	then 'Positive Communication'
		END AS  StrongestActivityType			
	,	lact.InteractionDate AS ActivityDate							
	,	lact.AttemptFlag
	,	lact.DialerUnsuccessfulAttemptFlag
	,	lact.ContactFlag							
	,	lact.PositiveRPCFlag						
	,	lact.NegativeRPCFlag							
	,	lact.RestRPCFlag 							
	,	app.ApplicationKey
	,	CAST(app.ApplicationDateCreated AS DATE) AS ApplicationDateCreated 
	,	app.ApplicationActivationDate
	,	CASE WHEN app.AccountImplementationStatus = 'Implemented'		THEN 'Υλοποιήσεις' 
			 WHEN app.AccountImplementationStatus = 'Not Implemented' 
			  AND app.ApplicationImplementationStatus <> 'Active'		THEN 'Ακυρωμένες/ Ολοκληρωμένες χωρίς Υλοποίηση'
																			ELSE app.StepCategory
		END		AS StepCategory
	,	app.StepCategoryOwner
	,	app.StepSubCategory
	,	app.StepSubCategoryOwner
	,	app.CreditInvolvementStatus
	,	app.CurrentStep
	,	app.ApplicationAssigneeGroupName
	,	CAST(app.CurrentStepAssignmentDate AS DATE)	AS CurrentStepAssignmentDate				
	,	CAST(app.AppSubmittedDate AS DATE)		AS ApplicationAssessmentDate
	,	app.ApplicationImplementationStatus
	,	app.AccountImplementationStatus
	,   CASE WHEN CAST(app.AppPreApprovedDate AS DATE) <= {run_date!r} THEN 1 ELSE 0 END	AS ApprovedFlag
	,	CAST(app.AppPreApprovedDate AS DATE) AS ApplicationApprovedDate
	,   CASE WHEN CAST(app.AppRejectedDate	AS DATE) <= {run_date!r} THEN 1 ELSE 0 END	AS RejectedFlag
	,	CAST(app.AppRejectedDate	AS DATE) AS RejectedDate
	,   CASE WHEN CAST(app.CancelledDate AS DATE) <= {run_date!r}  THEN 1 ELSE 0 END	AS CancelledFlag
	,	CAST(app.CancelledDate AS DATE)	AS CancelledDate
	,	CAST(app.AppApprovedDate AS DATE) AS ContractSigningDate									
	,	app.ContractSigningPlace
	,	app.ContractSigningPlaceType
	,	CAST(app.AppFinalStepDate AS DATE) AS AppFinalStepDate
	,	app.CustomerMobilePhone
	,	app.ContractSigningMethod
	,	app.CalculationMode
	,	app.ImplementationDate
	,	app.NewAccountNumber
	,	app.CommunicationChannel
	,	app.UserCreatedChannel
	,	app.ApplicationUserCreated
	,	app.ApplicationUserCreatedFullName
	,	app.ClientResponse
	,	app.ModificationType
	,	app.SolutionType
	,	app.Segment
	,	app.DeclaredAffordability
	,	app.StatedAffordability
	,	app.IncomeUsedForCalculations
	,	CASE WHEN app.AccountImplementationStatus = 'Implemented' AND lower(app.ModificationType) NOT LIKE '_dpo' THEN 'FLDT' END	AS Source
  	, 	SoM.Entity
	,	lactExclDialer.InteractionTypeKey AS LastActivityTypeKey
	,	lactExclDialer.InteractionType	 AS LastActivityName
	,	lactExclDialer.InteractionDate	 AS LastActivityDate
	,	COALESCE(lactExclDialer.ActivityTypeGroup, '0. no contact - no attempt')	 AS ContactLoanActivityGroup
	,	COALESCE(dact.ActivityTypeGroup, '0. no contact - no attempt')	 AS ContactDebtorActivityGroup
	,	app.DownPaymentAmount
 	, 	app.InstallmentAmountCalculated	
  	, 	app.FirstPeriodTotalInterestRate
  	, 	app.SpreadOfFirstPeriod
  	, 	app.DownPaymentAmountRate
    ,	app.ModificationProduct
    ,	app.ImplementationStatusDescription
	,	RANK() OVER(PARTITION BY SoM.AgreementKey ORDER BY SoM.ExposureKey ASC) 		 AS Rank
	,	SUM(SoMGBVEUR) OVER(PARTITION BY SoM.Entity) CustomerTotalSoMGBVEUR
	,	SUM(case when COALESCE(app.ApplicationImplementationStatus,'') = 'Active' then 1 else 0 end) OVER(PARTITION BY SoM.Entity) CustomerCountActiveApps 
    , 	SUM(CASE WHEN app.AccountImplementationStatus = 'Implemented' AND lower(app.ModificationType) NOT LIKE '_dpo'	AND CAST(app.ImplementationDate AS DATE) > {som_date!r} AND CAST(app.ImplementationDate AS DATE) <= {run_date!r} THEN 1 ELSE 0 END) OVER(PARTITION BY SoM.Entity) CustomerCountImplementedApps
    , SUM(CASE WHEN pip.SoMPipelineFlag = 1 AND CAST(app.CancelledDate	AS DATE) <= {run_date!r} AND CAST(app.CancelledDate	AS DATE) > {som_date!r} THEN 1 ELSE 0 END) OVER(PARTITION BY SoM.Entity) CustomerCountCancelledWithinMonthApps
FROM perimeter_v5 AS SoM --`{env_prefix}-cdh-delivery`.repoint.int_haps_ModificationsPerformance_Perimeter	 AS SoM
LEFT JOIN FLDTApplications				AS app   ON app.AccountNumber =SoM.ExposureKey AND app.Rnk = 1
LEFT JOIN ApplicationsPipeline 			AS pip 	ON SoM.ExposureKey = pip.AccountNumber
LEFT JOIN loanactivitiesMtD		AS lact  ON lact.ExposureKeyPrefix = SoM.ExposureKey and lact.rank = 1
LEFT JOIN loanactivitiesMtD		AS lactExclDialer  ON lactExclDialer.ExposureKeyPrefix = SoM.ExposureKey and lactExclDialer.rankExcludingDialer = 1 AND lactExclDialer.ActivityPriority <> 1.15
-- LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_haps_modificationsperformance_InteractionsMtD		AS lact  ON lact.ExposureKeyPrefix = SoM.ExposureKey and lact.rank = 1
-- LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_haps_modificationsperformance_InteractionsMtD		AS lactExclDialer  ON lactExclDialer.ExposureKeyPrefix = SoM.ExposureKey and lactExclDialer.rankExcludingDialer = 1 AND lactExclDialer.ActivityPriority <> 1.15
LEFT JOIN StrongestDebtorActivityMtD	AS dact  ON dact.Entity = SoM.Entity

'''
)

Perimeter_v6.createOrReplaceTempView("Perimeter_v6")

##### DebtorMetrics

In [0]:
# DebtorMetrics = spark.sql(
# f'''
# SELECT Entity
#     , SUM(SoMGBVEUR) SoMGBVEUR
# 		,	SUM(case when COALESCE(ApplicationImplementationStatus,'') = 'Active' then 1 else 0 end) CountActive 
#     , SUM(CASE WHEN AccountImplementationStatus = 'Implemented' AND lower(ModificationType) NOT LIKE '_dpo'	AND CAST(ImplementationDate AS DATE) > {som_date!r} AND CAST(ImplementationDate AS DATE) <= {run_date!r}	THEN 1 ELSE 0 END) CountImplemented
#     , SUM(CASE WHEN SoMPipelineFlag = 1 AND CancelledWithinMonthFlag = 1 THEN 1 ELSE 0 END) CountCancelledWithinMonth
# FROM Perimeter_v6
# GROUP BY Entity
# '''
# )

# DebtorMetrics.createOrReplaceTempView("DebtorMetrics")

##### NonFLDTModsQtD

In [0]:
NonFLDTModsQtD = spark.sql(
f'''
select AccountNumber, sum(cast(DownPaymentAmount as decimal(19,2))) as DownPaymentAmount
FROM  `{env_prefix}-reportinputfiles`.repoint.haps_nonfldtmodifications AS m
INNER JOIN (SELECT BusinessDate, max(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-reportinputfiles`.repoint.haps_nonfldtmodifications GROUP BY BusinessDate) its ON m.IngestionTimeStamp = its.IngestionTimeStamp AND m.BusinessDate = its.BusinessDate
WHERE YEAR(ReferenceDateEOM) = YEAR({LastDateOfMonth!r}) AND
DATE_PART('quarter', ReferenceDateEOM) = DATE_PART('quarter', {LastDateOfMonth!r}) AND
CAST(ReferenceDateEOM AS DATE) <= {LastDateOfMonth!r}		
group by AccountNumber
'''
)

NonFLDTModsQtD.createOrReplaceTempView("NonFLDTModsQtD")

##### DPOQtD

In [0]:
DPOQtD = spark.sql(
f'''
select AccountNumber, sum(cast(CashAmount as decimal(19,2))) as CashAmount
FROM  `{env_prefix}-reportinputfiles`.repoint.haps_dpo m
INNER JOIN (SELECT BusinessDate, max(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-reportinputfiles`.repoint.haps_dpo GROUP BY BusinessDate) its ON m.IngestionTimeStamp = its.IngestionTimeStamp AND m.BusinessDate = its.BusinessDate
WHERE YEAR(ReferenceDateEOM) = YEAR({LastDateOfMonth!r}) AND
DATE_PART('quarter', ReferenceDateEOM) = DATE_PART('quarter', {LastDateOfMonth!r}) AND
CAST(ReferenceDateEOM AS DATE) <= {LastDateOfMonth!r}	
group by AccountNumber
'''
)

DPOQtD.createOrReplaceTempView("DPOQtD")

##### Perimeter_v9

In [0]:
Perimeter_v9 = spark.sql(
f'''
SELECT DISTINCT
	    SoM.*
        ,   SoM.CustomerTotalSoMGBVEUR
        ,   CASE WHEN SoM.CustomerCountImplementedApps > 0           THEN '1. Implemented'
                 WHEN SoM.CustomerCountActiveApps > 0                THEN '2. Positive - Active Pipeline'
                 WHEN SoM.CustomerCountCancelledWithinMonthApps > 0  THEN '3. Negative With Cancelled Application'
                 WHEN SOM.ContactDebtorActivityGroup IN ('5.1.1 Positive to modify with Application','5.1.2 Positive to modify - no Application')       THEN '4. Positive to Modify - No Application'
                 WHEN SOM.ContactDebtorActivityGroup = '5.1.3 Positive to modify - pending documents '      THEN '5. Positive to Modify - Pending Documents'
                 WHEN SOM.ContactDebtorActivityGroup = '4.1 Negative to Modify'         THEN '6. Negative to Modify'
                 WHEN SOM.ContactDebtorActivityGroup = '4.3 Rest Refusals'              THEN '7. Rest Refusals'
                 WHEN SOM.ContactDebtorActivityGroup = '4.2 Refuse to Pay'              THEN '8. Refuse to Pay'
                 WHEN SOM.ContactDebtorActivityGroup = '3. Rest RPC Communications'      THEN '9. Rest RPC Communications'
                 WHEN SOM.ContactDebtorActivityGroup IN ('2.2.2  Deceased','2.2.1 Rest No RPC Contacts','2.1.1 Denial of identification')     THEN '10. Contact - Not RPC'
                 WHEN SOM.ContactDebtorActivityGroup IN ('1.2.1 no contact - attempt','1.2.2 no contact - attempt untraceable','1.2.3 Deceased', '1.1.1 no contact - no attempt')     THEN '11. No Contact'
                 end as CustomerCategory
	,   CASE WHEN d.AccountNumber IS NOT NULL                                 THEN 'Non FLDT'
             WHEN M.AccountNumber IS NOT NULL AND SOM.IsActiveFlag  = false         THEN 'Non FLDT'
        ELSE SoM.Source
        end as  Source_v2
	,   CASE WHEN d.AccountNumber IS NOT NULL AND (d.ImplementationType LIKE '%DPO%' or d.ImplementationType = 'Collateral change') 
                      AND cast(d.ModificationAmount AS decimal(19,2)) IS NOT NULL AND cast(d.ModificationAmount AS decimal(19,2)) <> 0      THEN 1
             WHEN M.AccountNumber IS NOT NULL AND SOM.IsActiveFlag  = false          THEN 1
        ELSE SoM.ModifiedWithinMonthFlag
        end as  ModifiedWithinMonthFlag_v2
	,   CASE WHEN dQtD.AccountNumber IS NOT NULL                                   THEN 1
             WHEN MQtD.AccountNumber IS NOT NULL AND SOM.IsActiveFlag  = false          THEN 1
        ELSE SoM.ModifiedWithinQuarterFlag
        end as  ModifiedWithinQuarterFlag_v2
 	,   CASE WHEN M.AccountNumber IS NOT NULL AND SOM.IsActiveFlag  = false          THEN m.DownPaymentAmount
        ELSE SoM.DownPaymentAmount
        end as  DownPaymentAmount_v2
 	,   CASE WHEN D.AccountNumber IS NOT NULL                                   THEN D.CashAmount
        ELSE NULL
        end as  DPOCashAmount_v2        
	,   CASE WHEN D.AccountNumber IS NOT NULL                                   THEN try_to_date(D.ImplementationDate, 'd/M/yyyy')
             WHEN M.AccountNumber IS NOT NULL AND SOM.IsActiveFlag  = false           THEN m.ImplementationDate
        ELSE SoM.ImplementationDate
        end as  ImplementationDate_v2
	,   CASE WHEN D.AccountNumber IS NOT NULL                                   THEN D.ImplementationType
             WHEN M.AccountNumber IS NOT NULL AND SOM.IsActiveFlag  = false           THEN m.ImplementationType
        ELSE SoM.ModificationType
        end as  ModificationType_v2
	,   CASE WHEN D.AccountNumber IS NOT NULL                                   THEN D.ImplementationProduct
             WHEN M.AccountNumber IS NOT NULL AND SOM.IsActiveFlag  = false           THEN m.ImplementationProduct
        ELSE SoM.ModificationProduct
        end as  ModificationProduct_v2
	,   CASE WHEN D.AccountNumber IS NOT NULL                                               THEN 'non FLDT'
             WHEN M.AccountNumber IS NOT NULL AND SOM.IsActiveFlag  = false             THEN 'DPO'
             ELSE NULL
        end as AuxModType           
        ,   CASE WHEN SOM.ModifiedWithinMonthFlag = 1 AND SOM.AccountImplementationStatus = 'Implemented' AND SOM.DownPaymentAmount > 0     then 1 
                WHEN M.AccountNumber IS NOT NULL AND SOM.IsActiveFlag  = false   and m.DownPaymentAmount > 0                    then 1 
                ELSE 0
        end as  DownPaymentAmountMonthFlag
        ,   CASE WHEN  d.AccountNumber IS NOT NULL AND CAST(d.CashAmount AS decimal(19,2)) > 0                    then 1 
                ELSE 0
        end as  DPOCashAmountMonthFlag
        ,   CASE WHEN SOM.ModifiedWithinQuarterFlag = 1 AND SOM.AccountImplementationStatus = 'Implemented' AND SOM.DownPaymentAmount > 0     then 1 
                WHEN MQtD.AccountNumber IS NOT NULL AND SOM.IsActiveFlag  = false  and MQtD.DownPaymentAmount > 0                    then 1 
                ELSE 0
        end as  DownPaymentAmountQuarterFlag
        ,   CASE WHEN  DQtD.AccountNumber IS NOT NULL AND DQtD.CashAmount > 0                    then 1 
                ELSE 0
        end as  DPOCashAmountQuarterFlag   
        ,    CASE WHEN d.AccountNumber IS NOT NULL AND CAST(d.ModificationAmount AS decimal(19,2)) > 0  THEN CAST(d.ModificationAmount AS decimal(19,2)) 
                 WHEN M.AccountNumber IS NOT NULL AND SOM.IsActiveFlag  = false and CAST(M.ImplementationAmount as decimal(19,2))>0    THEN CAST(M.ImplementationAmount as decimal(19,2))
         ELSE CAST ( SOM.SoMGBVEur AS decimal(19,2))
         end as  SoMGBVEur_v2
FROM Perimeter_v6 SoM
LEFT JOIN NonFLDTModsMtD as M ON RIGHT(SoM.ExposureKey, LEN(SoM.ExposureKey)-3) = CASE WHEN LEFT(CAST(M.AccountNumber AS STRING),1) RLIKE '[A-Z]' THEN RIGHT(CAST(M.AccountNumber AS STRING), LEN(CAST(M.AccountNumber AS STRING))-3) ELSE CAST(M.AccountNumber AS STRING) END
LEFT JOIN DPOMtD as D ON RIGHT(SoM.ExposureKey, LEN(SoM.ExposureKey)-3) = CASE WHEN LEFT(CAST(D.AccountNumber AS STRING),1) RLIKE '[A-Z]' THEN RIGHT(CAST(D.AccountNumber AS STRING), LEN(CAST(D.AccountNumber AS STRING))-3) ELSE CAST(D.AccountNumber AS STRING) END
LEFT JOIN NonFLDTModsQtD as MQtD ON RIGHT(SoM.ExposureKey, LEN(SoM.ExposureKey)-3) = CASE WHEN LEFT(CAST(MQtD.AccountNumber AS STRING),1) RLIKE '[A-Z]' THEN RIGHT(CAST(MQtD.AccountNumber AS STRING), LEN(CAST(MQtD.AccountNumber AS STRING))-3) ELSE CAST(MQtD.AccountNumber AS STRING) END
LEFT JOIN DPOQtD as DQtD ON RIGHT(SoM.ExposureKey, LEN(SoM.ExposureKey)-3) = CASE WHEN LEFT(CAST(DQtD.AccountNumber AS STRING),1) RLIKE '[A-Z]' THEN RIGHT(CAST(DQtD.AccountNumber AS STRING), LEN(CAST(DQtD.AccountNumber AS STRING))-3) ELSE CAST(DQtD.AccountNumber AS STRING) END
-- LEFT JOIN DebtorMetrics as DA ON SoM.Entity = DA.Entity
'''
)

Perimeter_v9.createOrReplaceTempView("Perimeter_v9")

In [0]:
AgentMapping = spark.sql(
f'''
SELECT  cast(UserID as int) AS UserID
      , UserName
      , FullName
      , AM
      , ActivityUserName
FROM `{env_prefix}-reportinputfiles`.repoint.haps_AgentMapping a
INNER JOIN (SELECT BusinessDate, max(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-reportinputfiles`.repoint.haps_AgentMapping GROUP BY BusinessDate) its ON a.IngestionTimeStamp = its.IngestionTimeStamp AND a.BusinessDate = its.BusinessDate
WHERE a.BusinessDate = (SELECT max(BusinessDate) FROM `{env_prefix}-reportinputfiles`.repoint.haps_AgentMapping WHERE BusinessDate <= {run_date!r})
'''
)

AgentMapping.createOrReplaceTempView("AgentMapping")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT  			
 CAST ( {run_date!r} AS date) AS BusinessDate
, CAST ( A.PortfolioName AS string) AS PortfolioName
, CAST ( A.CDI AS string) AS CDI
, CAST ( A.PartyTaxRegistrationNumber AS string) AS PartyTaxRegistrationNumber
, CAST (C0.LegalNameGR AS string) AS DebtorName
, CAST ( c0.PartyPostalCode AS string) AS PostalCode
, CAST ( c0.PartyRegion AS string) AS Region
, CAST ( A.AgreementKey AS string) AS AgreementKey
, CAST ( A.ExposureKey AS string) AS ExposureKey
, CAST ( A.AssetSegment AS string) AS AssetSegment
, CAST ( A.ProductKey AS string) AS ProductKey
, CAST ( A.SoMGBVEur_V2 AS decimal(19,2)) AS SoMGBVEur
, CAST ( A.SoMOnBalanceAmountEur AS decimal(19,2)) AS SoMOnBalanceAmountEur
, CAST ( A.SoMBucket AS string) AS SoMBucket
, CAST ( A.SoMExposureStatus AS string) AS SoMExposureStatus
, CAST ( A.IsActiveFlag AS string) AS IsActiveFlag
, CAST ( A.ClosingDate AS date) AS ClosingDate
, CAST ( MtD.Bucket AS string) AS MtDBucket
, CAST ( l3869.LawStatus AS string) AS L3869Status
, CAST (alps.LegalProcessStatusLevel2 AS string) AS LegalProcessStatusLevel2      
, CAST (alps.OperationalLegalStatusLevel3 AS string) AS OperationalLegalStatusLevel3
, CAST (alps.LegalProcessStatusLastChangeDate AS date) AS LegalProcessStatusLastChangeDate
, CAST ( A.SegmentationBusinessDate AS date) AS SegmentationBusinessDate
, CAST ( A.OfficialSegment AS string) AS OfficialSegment
, CAST ( A.PayerFlagEntity AS string) AS PayerFlagEntity
, CAST ( A.BorrowerTotalGBVRange AS string) AS BorrowerTotalGBVRange
, CAST ( A.SecuredUnsecuredAccount AS string) AS SecuredUnsecuredAccount
, CAST ( a.HighBalanceFlag AS smallint) AS HighBalanceFlag
, CAST ( A.AssignmentSegmentDetailed AS string) AS AssignmentSegmentDetailed
, CAST ( A.CourseOfAction AS string) AS CourseOfAction
, CAST ( A.AssignmentChannel AS string) AS AssignmentChannel
, CAST ( A.AssignmentTeam AS string) AS AssignmentTeam
, CAST ( A.AssignmentAgent AS string) AS AssignmentAgent
, CAST ( IFNULL(A.SoMPipelineFlag,0) AS smallint) AS SoMPipelineFlag
, CAST ( IFNULL(A.NewApplicationFlag,0) AS smallint) AS NewApplicationFlag
, CAST ( IFNULL(A.ModifiedWithinMonthFlag_v2,0) AS smallint) AS ModifiedWithinMonthFlag
, CAST ( IFNULL(a.ActivePipelineFlag,0) AS smallint) AS ActivePipelineFlag
, CAST ( IFNULL(A.RejectedWithinMonthFlag,0) AS smallint) AS RejectedWithinMonthFlag
, CAST ( IFNULL(A.CancelledWithinMonthFlag,0) AS smallint) AS CancelledWithinMonthFlag
, CAST ( IFNULL(A.CompletedNotImplementedWithinMonthFlag,0) AS smallint) AS CompletedNotImplementedWithinMonthFlag
, CAST ( IFNULL(A.AppliedWithinQuarterFlag,0) AS smallint) AS AppliedWithinQuarterFlag
, CAST ( IFNULL(A.ModifiedWithinQuarterFlag_v2,0) AS smallint) AS ModifiedWithinQuarterFlag
, CAST ( A.AttributionChannel AS string) AS AttributionChannel
, CAST ( A.AttributionTeam AS string) AS AttributionTeam
, CAST ( A.AttributionAgent AS string) AS AttributionAgent
, CAST ( A.AssignmentChannelFinal AS string) AS FinalAssignmentChannel
, CAST ( A.AssignmentTeamFinal AS string) AS FinalAssignmentTeam
, CAST ( A.AssignmentAgentFinal AS string) AS FinalAssignmentAgent
, CAST ( A.StrongestActivityType AS string) AS StrongestActivityType
, CAST ( A.ActivityDate AS date) AS ActivityDate
, CAST ( IFNULL(A.AttemptFlag,0) AS smallint) AS AttemptFlag
, CAST ( IFNULL(A.ContactFlag,0) AS smallint) AS ContactFlag
, CAST ( IFNULL(A.PositiveRPCFlag,0) AS smallint) AS PositiveRPCFlag
, CAST ( IFNULL(A.NegativeRPCFlag,0) AS smallint) AS NegativeRPCFlag
, CAST ( IFNULL(A.RestRPCFlag,0) AS smallint) AS RestRPCFlag
, CAST ( A.ApplicationKey AS string) AS ApplicationKey
, CAST ( A.ApplicationDateCreated AS date) AS ApplicationDateCreated
, CAST ( A.ApplicationActivationDate AS date) AS ApplicationActivationDate
, CAST ( A.StepCategory AS string) AS StepCategory
, CAST ( A.StepCategoryOwner AS string) AS StepCategoryOwner
, CAST ( A.StepSubCategory AS string) AS StepSubCategory
, CAST ( A.StepSubCategoryOwner AS string) AS StepSubCategoryOwner
, CAST ( A.CreditInvolvementStatus AS string) AS CreditInvolvementStatus
, CAST ( A.CurrentStep AS string) AS CurrentStep
, CAST ( A.ApplicationAssigneeGroupName AS string) AS ApplicationAssigneeGroupName
, CAST ( A.CurrentStepAssignmentDate AS date) AS CurrentStepAssignmentDate
, CAST ( A.ApplicationAssessmentDate AS date) AS ApplicationAssessmentDate
, CAST ( A.ApplicationImplementationStatus AS string) AS ApplicationImplementationStatus
, CAST ( A.AccountImplementationStatus AS string) AS AccountImplementationStatus
, CAST ( A.ApprovedFlag AS smallint) AS ApprovedFlag
, CAST ( A.ApplicationApprovedDate AS date) AS ApplicationApprovedDate
, CAST ( A.RejectedFlag AS smallint) AS RejectedFlag
, CAST ( A.RejectedDate AS date) AS RejectedDate
, CAST ( A.CancelledFlag AS smallint) AS CancelledFlag
, CAST ( A.CancelledDate AS date) AS CancelledDate
, CAST ( A.ContractSigningDate AS date) AS ContractSigningDate
, CAST ( A.ContractSigningPlace AS string) AS ContractSigningPlace
, CAST ( A.ContractSigningPlaceType AS string) AS ContractSigningPlaceType
, CAST ( A.AppFinalStepDate AS date) AS AppFinalStepDate
, CAST ( A.CustomerMobilePhone AS string) AS CustomerMobilePhone
, CAST ( A.ContractSigningMethod AS string) AS ContractSigningMethod
, CAST ( A.CalculationMode AS string) AS CalculationMode
, CAST ( A.ImplementationDate_v2 AS date) AS ImplementationDate
, CAST ( A.NewAccountNumber AS string) AS NewAccountNumber
, CAST ( A.CommunicationChannel AS string) AS CommunicationChannel
, CAST ( A.UserCreatedChannel AS string) AS UserCreatedChannel
, CAST ( A.ApplicationUserCreatedFullName AS string) AS ApplicationUserCreatedFullName
, CAST ( A.ApplicationUserCreated AS string) AS ApplicationUserCreated
, CAST ( A.ClientResponse AS string) AS ClientResponse
, CAST ( A.ModificationType_v2 AS string) AS ModificationType
, CAST ( A.SolutionType AS string) AS SolutionType
, CAST ( A.Segment AS string) AS Segment
, CAST ( A.DeclaredAffordability AS decimal(19,2)) AS DeclaredAffordability
, CAST ( A.StatedAffordability AS decimal(19,2)) AS StatedAffordability
, CAST ( A.IncomeUsedForCalculations AS decimal(19,2)) AS IncomeUsedForCalculations
, CAST ( A.Source_V2 AS string) AS Source
, CAST ( A.Entity AS string) AS Entity
, CAST ( A.LastActivityTypeKey AS string) AS LastActivityTypeKey
, CAST ( A.LastActivityName AS string) AS LastActivityName
, CAST ( A.LastActivityDate AS date) AS LastActivityDate
, CAST ( actt.TotalActivities1stWeek AS smallint) AS TotalActivities1stWeek
, CAST ( actt.TotalActivities2ndWeek AS smallint) AS TotalActivities2ndWeek
, CAST ( actt.TotalActivities3rdWeek AS smallint) AS TotalActivities3rdWeek
, CAST ( actt.TotalActivities4thWeek AS smallint) AS TotalActivities4thWeek
, CAST ( actt.TotalActivitiesMtD AS smallint) AS TotalActivitiesMtD
, CAST ( lpay.PaymentAmountEUR AS decimal(19,2)) AS LastPaymentAmountEUR
, CAST ( lpay.PaymentActualDate AS date) AS LastPaymentDate
, CAST ( A.CustomerTotalSoMGBVEUR AS decimal(19,2)) AS CustomerTotalSoMGBVEUR
, CAST ( A.ContactLoanActivityGroup AS string) AS ContactLoanActivityGroup
, CAST ( A.ContactDebtorActivityGroup AS string) AS ContactDebtorActivityGroup
, CAST ( CASE WHEN dftf.FtFActivity = 1 THEN 1 ELSE 0 END AS smallint) AS IntramonthFtFMeeting
, CAST ( A.CustomerCategory AS string) AS CustomerCategory
, CAST ( CASE WHEN a.AuxModType = 'non FLDT'     then a.DownPaymentAmount_v2
              WHEN a.Rank = 1 THEN a.DownPaymentAmount END AS decimal(19,2)) AS DownPaymentAmount
, CAST ( A.DPOCashAmount_v2 AS decimal(19,2)) AS DPOCashAmount
, CAST ( m1.UserID AS string) AS AssignmentAgentUserID
, CAST ( m1.UserName AS string) AS AssignmentAgentUserName
, CAST ( m1.AM AS string) AS AssignmentAgentAM
, CAST ( m2.UserID AS string) AS FinalAssignmentAgentUserID
, CAST ( m2.UserName AS string) AS FinalAssignmentAgentUserName
, CAST ( m2.AM AS string) AS FinalAssignmentAgentAM
, CAST ( m3.UserID AS string) AS AttributionAgentUserID
, CAST ( m3.UserName AS string) AS AttributionAgentUserName
, CAST ( m3.AM AS string) AS AttributionAgentAM
-- , CAST ( IFNULL(m1.UserID,m2.UserID) AS string) AS AssignmentAgentUserID
-- , CAST ( IFNULL(m1.UserName,m2.UserName) AS string) AS AssignmentAgentUserName
-- , CAST ( IFNULL(m1.AM,m2.AM) AS string) AS AssignmentAgentAM
-- , CAST ( IFNULL(m3.UserID,m4.UserID) AS string) AS FinalAssignmentAgentUserID
-- , CAST ( IFNULL(m3.UserName,m4.UserName) AS string) AS FinalAssignmentAgentUserName
-- , CAST ( IFNULL(m3.AM,m4.AM) AS string) AS FinalAssignmentAgentAM
-- , CAST ( IFNULL(m5.UserID,m6.UserID) AS string) AS AttributionAgentUserID
-- , CAST ( IFNULL(m5.UserName,m6.UserName) AS string) AS AttributionAgentUserName
-- , CAST ( IFNULL(m5.AM,m6.AM) AS string) AS AttributionAgentAM
, CAST ( A.DialerUnsuccessfulAttemptFlag AS smallint) AS DialerUnsuccessfulAttemptFlag
, CAST ( A.DownPaymentAmountMonthFlag AS smallint) AS DownPaymentAmountMonthFlag
, CAST ( A.DPOCashAmountMonthFlag AS smallint) AS DPOCashAmountMonthFlag
, CAST ( A.DownPaymentAmountQuarterFlag AS smallint) AS DownPaymentAmountQuarterFlag
, CAST ( A.DPOCashAmountQuarterFlag AS smallint) AS DPOCashAmountQuarterFlag
, CAST ( a.InstallmentAmountCalculated AS decimal(19,2)) AS InstallmentAmountCalculated
, CAST ( a.FirstPeriodTotalInterestRate AS decimal(19,2)) AS FirstPeriodTotalInterestRate
, CAST ( a.SpreadOfFirstPeriod AS decimal(19,2)) AS SpreadOfFirstPeriod
, CAST ( a.DownPaymentAmountRate AS decimal(19,2)) AS DownPaymentAmountRate
, CAST ( ModificationProduct_v2 AS string) AS ModificationProduct
, CAST ( ImplementationStatusDescription AS string) AS ImplementationStatusDescription
, CAST ( getdate() AS timestamp) AS InsertTimeStamp
FROM Perimeter_v9 a
LEFT JOIN AgentMapping m1 ON ifnull(m1.ActivityUserName,m1.FullName) = a.AssignmentAgent 
LEFT JOIN AgentMapping m2 ON ifnull(m2.ActivityUserName,m2.FullName) = a.AssignmentAgentFinal 
LEFT JOIN AgentMapping m3 ON ifnull(m3.ActivityUserName,m3.FullName) = a.AttributionAgent 

-- LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.securitized_AgentMapping m1 ON m1.ActivityUserName = a.AssignmentAgent 
-- LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.securitized_AgentMapping m2 ON m2.FullName = a.AssignmentAgent 
-- LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.securitized_AgentMapping m3 ON m3.ActivityUserName = a.AssignmentAgentFinal 
-- LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.securitized_AgentMapping m4 ON m4.FullName = a.AssignmentAgentFinal 
-- LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.securitized_AgentMapping m5 ON m5.ActivityUserName = a.AttributionAgent 
-- LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.securitized_AgentMapping m6 ON m6.FullName = a.AttributionAgent 
LEFT JOIN dPartyMtd			as c0 on c0.cdi = a.cdi 
LEFT JOIN AgreementLPS 			as aLPS on aLPS.AgreementKey = a.AgreementKey
LEFT JOIN AccountsQualcoMtD		as MtD	on MtD.ExposureKey = RIGHT(a.ExposureKey, LEN(a.ExposureKey) - 3)
LEFT JOIN L3869Status			AS l3869 ON l3869.CDI = a.CDI
LEFT JOIN TotalActivitiesMtD		AS actt  ON actt.ExposureKeyPrefix = a.ExposureKey
LEFT JOIN LastPayment			AS lpay  ON lpay.ExposureKey = a.ExposureKey
LEFT JOIN DebtorFtFActivityMtD		AS dftf  ON dftf.Entity = a.Entity
'''
)

final_df.createOrReplaceTempView("final_df")

In [0]:
%python
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
ModificationsPerformance = spark.sql(
f'''
select * from `{env_prefix}-cdh-delivery`.repoint.haps_ModificationsPerformance
where BusinessDate = {run_date!r}
'''
)
ModificationsPerformance.createOrReplaceTempView("ModificationsPerformance")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/HAPS/ModificationsPerformance/" 
file_name = "ModificationsPerformance_" +run_date+ ".csv"
ModificationsPerformance.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}") 
rename_and_cleanup_files(output_path, file_name)